In [1]:
import pandas as pd
import numpy as np

# 1) 读 submission template（作为 wq_df：lat/lon/date）
sub = pd.read_csv("../resources/data/submission_template.csv")
sub["Sample Date"] = pd.to_datetime(sub["Sample Date"], dayfirst=True, errors="coerce")

# 2) 读 Landsat 两源 + TerraClimate
ls_api = pd.read_csv("../data/processed/landsat_api_validation.csv")                # 有 Index
ls_off = pd.read_csv("../resources/code/general/landsat_features_validation.csv")           # official
terra  = pd.read_csv("../resources/code/general/terraclimate_features_validation.csv")      # 含 lat/lon/date + 气候变量
terra["Sample Date"] = pd.to_datetime(terra["Sample Date"], errors="coerce")

# 3) 对齐 Landsat API 到 submission 行号
ls_api_aligned = ls_api.set_index("Index").reindex(range(len(sub)))

# 4) 用 “API 优先 + Official 回填” 合并 Landsat（最简实用版）
#    目标输出列：blue, green, red, nir08, swir16, swir22 (+ 可选 NDMI/MNDWI)
landsat = pd.DataFrame(index=range(len(sub)))

band_cols = ["blue", "green", "red", "nir08", "swir16", "swir22"]
for c in band_cols:
    landsat[c] = ls_api_aligned[c] if c in ls_api_aligned.columns else np.nan

# 用 official 回填 nir/green/swir16/swir22（你的 train 脚本里就是这么做的）
official_to_api = {"nir": "nir08", "green": "green", "swir16": "swir16", "swir22": "swir22"}
api_nan_mask = landsat["green"].isna()

for off_c, api_c in official_to_api.items():
    if off_c in ls_off.columns:
        fill_mask = api_nan_mask & ls_off[off_c].notna()
        landsat.loc[fill_mask, api_c] = ls_off.loc[fill_mask, off_c].values

# NDMI/MNDWI：如果 official 有就直接拿来（和你脚本一致）
for idx_col in ["NDMI", "MNDWI"]:
    if idx_col in ls_off.columns:
        landsat[idx_col] = ls_off[idx_col].values

# 5) 合并 TerraClimate（去掉重复 key）
terra_feats = terra.drop(columns=["Latitude", "Longitude", "Sample Date"], errors="ignore")

val_merged = pd.concat([sub.reset_index(drop=True),
                        landsat.reset_index(drop=True),
                        terra_feats.reset_index(drop=True)], axis=1)

print(val_merged.shape)
print(val_merged.columns)

(200, 15)
Index(['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity',
       'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'blue',
       'green', 'red', 'nir08', 'swir16', 'swir22', 'NDMI', 'MNDWI', 'pet'],
      dtype='object')


In [3]:
import pandas as pd

# 1) 你现在这个 200×15 的表，假设叫 val_merged

val_df=val_merged.copy()  # 先复制一份，保持原表不变
val_df = val_df.copy()
val_df["Sample Date"] = pd.to_datetime(val_df["Sample Date"], dayfirst=True, errors="coerce")

terra = pd.read_csv("../resources/code/general/terraclimate_features_validation.csv")
terra["Sample Date"] = pd.to_datetime(terra["Sample Date"], errors="coerce")

# 2) 建一个“月份键”用于对齐（和你训练脚本逻辑一致：月度匹配）
val_df["ym"] = val_df["Sample Date"].dt.to_period("M").astype(str)
terra["ym"]  = terra["Sample Date"].dt.to_period("M").astype(str)

# 3) TerraClimate 去掉重复键列，只保留特征（通常含 pet/ppt/tmax/tmin/q）
terra_feats = terra.drop(columns=["Sample Date"], errors="ignore")

# 4) 合并（Latitude/Longitude/ym）
val_merged = val_df.merge(
    terra_feats,
    on=["Latitude", "Longitude", "ym"],
    how="left",
    suffixes=("", "_terra")
)

# 5) 清理辅助列
val_merged.drop(columns=["ym"], inplace=True)

print(val_merged.shape)
print([c for c in ["pet","ppt","tmax","tmin","q"] if c in val_merged.columns])
print(val_merged[["pet","ppt","tmax","tmin","q"]].isna().mean())

(212, 17)
['pet']


KeyError: "['ppt', 'tmax', 'tmin', 'q'] not in index"

In [4]:
terra = pd.read_csv("../resources/code/general/terraclimate_features_validation.csv")
print(terra.shape)
print(terra.columns.tolist())

(200, 4)
['Latitude', 'Longitude', 'Sample Date', 'pet']


In [5]:
def load_terraclimate_dataset():
    """从 Planetary Computer 打开 TerraClimate Zarr 数据集。"""
    import pystac_client
    import planetary_computer as pc
    import xarray as xr

    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]
    ds = xr.open_dataset(asset.href, **asset.extra_fields["xarray:open_kwargs"])
    return ds

In [6]:
def fetch_terra_vars_vectorized(wq_df: pd.DataFrame, terra_df: pd.DataFrame, variables: list):
    """
    高速向量化方案：
    1. 一次性从 Zarr 中加载南非区域所有所需变量 (sel + where)。
    2. 用 xarray 的 sel(method='nearest') 做空间最近邻。
    3. 用 pandas 向量化做时间最近月匹配。
    """
    import xarray as xr

    print("  → 打开 TerraClimate Zarr …")
    ds = load_terraclimate_dataset()

    # 时间、空间裁剪（一次完成）
    # 注意：TerraClimate lat 坐标从正到负（降序），所以 slice 要大→小
    ds = ds[variables].sel(
        time=slice("2011-01-01", "2015-12-31"),
        lat=slice(-21.72, -35.18),
        lon=slice(14.97, 32.79),
    )
    print(f"  → 裁剪后 ds shape: { {k: ds[k].shape for k in variables} }")

    # 准备采样点日期
    dates = pd.to_datetime(wq_df["Sample Date"], dayfirst=True, errors="coerce")
    lats = wq_df["Latitude"].values
    lons = wq_df["Longitude"].values

    # 对每个采样点，找最近的网格月份
    # TerraClimate 是月度数据，每月1日。将采样日期 floor 到月初即可。
    month_starts = dates.dt.to_period("M").dt.to_timestamp()

    # 使用 xarray sel(method='nearest') 进行批量空间+时间查询
    # 需要构造 DataArray 坐标
    lat_da = xr.DataArray(lats, dims="sample")
    lon_da = xr.DataArray(lons, dims="sample")
    time_da = xr.DataArray(month_starts.values, dims="sample")

    print("  → 批量 sel(method='nearest') 查询中…")
    result = ds.sel(lat=lat_da, lon=lon_da, time=time_da, method="nearest")

    for var in variables:
        vals = result[var].values
        terra_df[var] = vals
        n_nan = np.isnan(vals).sum() if np.issubdtype(vals.dtype, np.floating) else 0
        print(f"    ✅ {var}: 提取完成 (NaN: {n_nan})")

    return terra_df

In [7]:
def merge_landsat_sources(wq_df: pd.DataFrame, official_df: pd.DataFrame, api_df: pd.DataFrame) -> pd.DataFrame:
    """
    合并两个 Landsat 数据源，最大化覆盖率：
    - 优先使用 API 数据（Phase 4 纯净云掩码，精确时间匹配）
    - API 缺失时回退到 Official 数据
    - 双源均缺失的行保留 NaN（仅 ~2.6%）
    
    统一输出列名为 ensemble_model.py 所用的 API 列名：
    blue, green, red, nir08, swir16, swir22
    """
    # Official 列名映射到 API 列名
    OFFICIAL_TO_API = {
        "nir": "nir08",
        "green": "green",
        "swir16": "swir16",
        "swir22": "swir22",
    }
    BAND_COLS = ["blue", "green", "red", "nir08", "swir16", "swir22"]
    
    # 和 wq_df 行索引对齐
    api_aligned = api_df.set_index("Index").reindex(range(len(wq_df)))
    
    # 用 API 的数据初始化
    result = pd.DataFrame(index=range(len(wq_df)))
    for col in BAND_COLS:
        if col in api_aligned.columns:
            result[col] = api_aligned[col].values
        else:
            result[col] = np.nan
    
    # 对于 API 缺失的行，用 Official 数据回填
    api_nan_mask = result["green"].isnull()
    n_api_nan = api_nan_mask.sum()
    
    filled_count = 0
    for api_col in BAND_COLS:
        # 找到 Official 中对应的列
        off_col = None
        for ok, av in OFFICIAL_TO_API.items():
            if av == api_col:
                off_col = ok
                break
        if off_col is None:
            # blue, red 在 Official 中不存在
            continue
        if off_col not in official_df.columns:
            continue
        
        # 仅在 API 缺失 & Official 有值时回填
        fill_mask = api_nan_mask & official_df[off_col].notna()
        result.loc[fill_mask, api_col] = official_df.loc[fill_mask, off_col].values
    
    # 同时用 Official 的 NDMI/MNDWI 回填（如果 API 没有这些列）
    for idx_col in ["NDMI", "MNDWI"]:
        if idx_col in official_df.columns:
            result[idx_col] = np.nan
            # Official 有值的行
            off_valid = official_df[idx_col].notna()
            # API 有数据的行可以重新算，但 Official-only 行需要直接用 Official 的值
            result.loc[off_valid, idx_col] = official_df.loc[off_valid, idx_col].values
    
    final_nan = result["green"].isnull().sum()
    recovered = n_api_nan - final_nan
    print(f"  Landsat 双源合并: API={len(wq_df)-n_api_nan}, Official回填={recovered}, 不可恢复NaN={final_nan} ({final_nan/len(wq_df)*100:.1f}%)")
    
    return result


# ── 3. 全量合并 + 清洗 ────────────────────────────────────────────────────────
def merge_and_clean(wq_df: pd.DataFrame, landsat_merged: pd.DataFrame, terra_df: pd.DataFrame) -> pd.DataFrame:
    """
    拼接 水质标签 + Landsat(双源合并) + TerraClimate，做基础清洗。
    
    关键决策（来自开发日志 Phase 4-7 的经验教训）：
    - Landsat NaN 保留不填充：XGBoost/LightGBM/CatBoost 均原生支持 NaN 路由，
      中位数填充会制造虚假"典型"光谱信号，对云遮蔽行不代表真实地表。
    - 目标变量 NaN 保留不填充：避免引入标签偏差。
    - TerraClimate 完整无缺（API 向量化拉取保证）。
    """
    assert len(wq_df) == len(landsat_merged) == len(terra_df), (
        f"行数不一致: wq={len(wq_df)}, landsat={len(landsat_merged)}, terra={len(terra_df)}"
    )

    # 验证 terra key 列一致
    assert np.allclose(wq_df["Latitude"].values, terra_df["Latitude"].values, atol=1e-4), \
        "terra Latitude 不对齐"
    assert np.allclose(wq_df["Longitude"].values, terra_df["Longitude"].values, atol=1e-4), \
        "terra Longitude 不对齐"

    # --- 拼接 ---
    terra_feats = terra_df.drop(columns=["Latitude", "Longitude", "Sample Date"], errors="ignore")

    merged = pd.concat([
        wq_df.reset_index(drop=True),
        landsat_merged.reset_index(drop=True),
        terra_feats.reset_index(drop=True),
    ], axis=1)

    print(f"合并后 shape: {merged.shape}")

    # --- 类型转换 ---
    merged["Sample Date"] = pd.to_datetime(merged["Sample Date"], dayfirst=True, errors="coerce")

    target_cols = ["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]
    # Rename the long DRP column if needed
    for c in merged.columns:
        if "Dissolved" in c and "Phosphorus" in c and c != "Dissolved Reactive Phosphorus":
            merged.rename(columns={c: "Dissolved Reactive Phosphorus"}, inplace=True)

    numeric_cols = [c for c in merged.columns if c not in ["Sample Date"]]
    for c in numeric_cols:
        merged[c] = pd.to_numeric(merged[c], errors="coerce")

    # --- 缺失值报告（不做中位数填充！）---
    print("\n缺失值统计：")
    missing = merged.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0])
    else:
        print("  无缺失")

    # 报告各类缺失
    feature_cols = [c for c in merged.columns if c not in target_cols + ["Sample Date", "Latitude", "Longitude"]]
    feat_nan = sum(merged[c].isnull().any() for c in feature_cols)
    if feat_nan > 0:
        print(f"\n  ℹ️  {feat_nan} 个特征列含 NaN → 保留原值，由树模型原生 NaN 路由处理")
        print("     (开发日志 Phase 4-7 已证实中位数填充会制造虚假信号，降低泛化能力)")

    for c in target_cols:
        if c in merged.columns and merged[c].isnull().any():
            n_miss = merged[c].isnull().sum()
            print(f"  ⚠ 目标列 {c} 有 {n_miss} 个 NaN（保留不填充）")

    print(f"\n最终 shape: {merged.shape}")
    total_nan = merged[feature_cols].isnull().sum().sum()
    total_cells = len(merged) * len(feature_cols)
    print(f"特征 NaN: {total_nan}/{total_cells} ({total_nan/total_cells*100:.2f}%)")
    return merged

In [ ]:
ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
GENERAL_DIR = os.path.join(ROOT, "resources", "code", "general")
DATA_DIR = os.path.join(ROOT, "resources", "data")
OUTPUT_DIR = os.path.join(ROOT, "data")

WQ_FILE = os.path.join(DATA_DIR, "water_quality_validation_dataset.csv")
LANDSAT_OFFICIAL = os.path.join(GENERAL_DIR, "landsat_features_validation.csv")
LANDSAT_API = os.path.join(ROOT, "data", "processed", "landsat_api_validation.csv")
TERRA_FILE = os.path.join(GENERAL_DIR, "terraclimate_features_validation.csv")
MERGED_OUT = os.path.join(OUTPUT_DIR, "merged_validation_data_clean.csv")

TERRA_VARS_NEEDED = ["ppt", "tmax", "tmin", "q"]  # q = runoff in TerraClimate

In [8]:
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--skip-api", action="store_true", help="跳过 TerraClimate API 拉取")
    args = parser.parse_args()

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1. 加载
    print("📂 加载数据…")
    wq_df = pd.read_csv(WQ_FILE)
    landsat_official = pd.read_csv(LANDSAT_OFFICIAL)
    landsat_api = pd.read_csv(LANDSAT_API)
    terra_df = pd.read_csv(TERRA_FILE)

    print(f"  water_quality    : {wq_df.shape}")
    print(f"  landsat_official : {landsat_official.shape} (NaN: {landsat_official['nir'].isnull().sum()})")
    print(f"  landsat_api      : {landsat_api.shape} (NaN: {landsat_api['green'].isnull().sum()})")
    print(f"  terraclimate     : {terra_df.shape}")

    # 2. 检查 & 拉取缺失气候变量
    if not args.skip_api:
        terra_df = fetch_missing_terra_vars(wq_df, terra_df)
    else:
        print("⏭  跳过 API 拉取（--skip-api）")

    # 3. Landsat 双源合并
    print("\n📡 Landsat 双源合并 (API-first, Official-fallback)…")
    landsat_merged = merge_landsat_sources(wq_df, landsat_official, landsat_api)

    # 4. 全量合并 & 清洗
    print("\n🔧 合并 & 清洗…")
    merged = merge_and_clean(wq_df, landsat_merged, terra_df)

    # 4. 输出
    merged.to_csv(MERGED_OUT, index=False)
    print(f"\n💾 已保存: {MERGED_OUT}")
    print("✅ 数据工程管线完成。")